# 04 - Run Judge (run 3): DBAF persona + assistant judge, categories D/B/A/F only

Third, separate judge run -- independent of notebooks 01 and 03 (reuses the cached raw
dataset from `data/raw/posts.parquet`, downloading it if that cache doesn't exist yet).

Scope for this run:

- **Categories**: D (Economics), B (Technology), A (Identity), F (Promotion) -- the four
  categories with the most `toxicity==4` posts in the dataset (254, 158, 59, 59
  respectively) -- see `config.DBAF_CATEGORIES`.
- **DBAF persona judge** (toxicity==4, 50 posts/category): same taxonomy as the AGE_ run --
  `sycophantic / malicious-manipulative / evil / other`.
- **DBAF assistant judge** (toxicity==0, 50 posts/category): the same canonical
  `AssistantJudgeResult` schema used by every run in this repo --
  `is_safe_for_assistant` always means "safe content for an assistant agent to post".

Results are written to `results/DBAF_results/DBAF_persona.jsonl` and
`results/DBAF_results/DBAF_assistant.jsonl`, each row also carrying the post's `content`
and `category` so the cache is self-contained for spot-checking.

Safe to interrupt and re-run: sampling is cached to `data/processed/dbaf_*sample.parquet`,
and already-judged post_ids are skipped on every run (see `ResultStore` in
`src/categorizer.py`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import config, dataset, prompts
from src.judge_schema import AgePersonaJudgeResult, AssistantJudgeResult
from src.categorizer import ResultStore, get_client, run_judge

## 1. Load posts (cached; downloads only if `data/raw/posts.parquet` is missing)

In [2]:
df = dataset.download_posts()
print(f"loaded {len(df)} rows")

loaded 44376 rows


## 2. Stratified samples (50/category, categories D/B/A/F only)

Cached to `data/processed/dbaf_persona_sample.parquet` and
`data/processed/dbaf_assistant_sample.parquet` -- re-running this cell will not
reshuffle the sample once cached.

In [3]:
dbaf_persona_sample = dataset.get_or_create_sample(
    df,
    name=config.DBAF_PERSONA_SAMPLE_NAME,
    toxicity=config.TOXICITY_MALICIOUS,
    per_category=config.DBAF_SAMPLES_PER_CATEGORY,
    categories=config.DBAF_CATEGORIES,
)
print(f"DBAF persona sample: {len(dbaf_persona_sample)} rows")
dbaf_persona_sample[config.COL_CATEGORY].value_counts().sort_index()

DBAF persona sample: 200 rows


topic_label
A    50
B    50
D    50
F    50
Name: count, dtype: int64

In [4]:
dbaf_assistant_sample = dataset.get_or_create_sample(
    df,
    name=config.DBAF_ASSISTANT_SAMPLE_NAME,
    toxicity=config.TOXICITY_SAFE,
    per_category=config.DBAF_SAMPLES_PER_CATEGORY,
    categories=config.DBAF_CATEGORIES,
)
print(f"DBAF assistant sample: {len(dbaf_assistant_sample)} rows")
dbaf_assistant_sample[config.COL_CATEGORY].value_counts().sort_index()

DBAF assistant sample: 200 rows


topic_label
A    50
B    50
D    50
F    50
Name: count, dtype: int64

## 3. Run judges

Each result row also stores `content` and `category` (via `content_key`/`category_key`),
so `results/DBAF_results/*.jsonl` is self-contained -- no separate join needed for EDA.

In [5]:
client = get_client()

dbaf_persona_store = ResultStore(config.DBAF_RESULTS_DIR / f"{config.DBAF_PERSONA_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=dbaf_persona_store,
    rows=dbaf_persona_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.AGE_PERSONA_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_age_persona_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AgePersonaJudgeResult,
    content_key=config.COL_CONTENT,
    category_key=config.COL_CATEGORY,
)
print(f"DBAF persona judge: {len(dbaf_persona_store.seen_ids())} successful rows cached")

judging (AgePersonaJudgeResult): 100%|██████████| 200/200 [09:41<00:00,  2.91s/it]

DBAF persona judge: 200 successful rows cached


In [6]:
dbaf_assistant_store = ResultStore(config.DBAF_RESULTS_DIR / f"{config.DBAF_ASSISTANT_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=dbaf_assistant_store,
    rows=dbaf_assistant_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.ASSISTANT_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_assistant_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AssistantJudgeResult,
    content_key=config.COL_CONTENT,
    category_key=config.COL_CATEGORY,
)
print(f"DBAF assistant judge: {len(dbaf_assistant_store.seen_ids())} successful rows cached")

judging (AssistantJudgeResult): 100%|██████████| 200/200 [10:44<00:00,  3.22s/it]

DBAF assistant judge: 187 successful rows cached


## 4. Error sanity check

Error rows are retried automatically on the next run of this notebook.

In [7]:
for name, store in [("DBAF persona", dbaf_persona_store), ("DBAF assistant", dbaf_assistant_store)]:
    rows = store.load_all_rows()
    errors = [r for r in rows if "error" in r]
    print(f"{name}: {len(rows)} total rows, {len(errors)} errors, {len(rows) - len(errors)} successful")
    for e in errors[:5]:
        print("  ", e)

DBAF persona: 200 total rows, 0 errors, 200 successful
DBAF assistant: 200 total rows, 13 errors, 187 successful
   {'post_id': '7840bb63-63e0-40a1-ac51-ef5d7f0ee78b', 'error': "validation_error: 3 validation errors for AssistantJudgeResult\nis_safe_for_assistant\n  Field required [type=missing, input_value={'description': 'Output o...40a1-ac51-ef5d7f0ee78b'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nconfidence\n  Field required [type=missing, input_value={'description': 'Output o...40a1-ac51-ef5d7f0ee78b'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nrationale\n  Field required [type=missing, input_value={'description': 'Output o...40a1-ac51-ef5d7f0ee78b'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing"}
   {'post_id': '088c4bf2-b771-446d-9e58-d4c76970d899', 'error': "validation_error: 3 validation errors for AssistantJudgeResult\n

## 5. Summary: personas + safety per category

Uses the `category` field stored directly in each result row -- no join needed.

In [8]:
dbaf_persona_rows = [r for r in dbaf_persona_store.load_all_rows() if "error" not in r]
dbaf_persona_df = pd.DataFrame(dbaf_persona_rows)

persona_by_category = (
    dbaf_persona_df.groupby(["category", "persona"]).size().unstack(fill_value=0)
)
print("Persona counts per category:")
persona_by_category

Persona counts per category:


persona,evil,malicious-manipulative,other
category,,,
A,27,20,3
B,0,38,12
D,1,45,4
F,0,38,12


In [9]:
dbaf_assistant_rows = [r for r in dbaf_assistant_store.load_all_rows() if "error" not in r]
dbaf_assistant_df = pd.DataFrame(dbaf_assistant_rows)

safety_by_category = (
    dbaf_assistant_df.groupby("category")["is_safe_for_assistant"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "safe_count", "count": "total"})
)
safety_by_category["safe_rate"] = safety_by_category["safe_count"] / safety_by_category["total"]
print("Assistant-safety rate per category:")
safety_by_category

Assistant-safety rate per category:


,safe_count,total,safe_rate
category,,,
A,39,47,0.829787
B,44,48,0.916667
D,24,44,0.545455
F,36,48,0.750000
